# From Retrieval to Citation: The Effectiveness of Generative Engine Optimization Beyond the Generation Stage
#### RQ: How do content-based GEO edits affect the retrieval and generation stages of GSEs?

---
**Authors:** Leonard Rampf, Niklas Keckeisen

**Advisors:** Michail Batikas

**Submission Date:** 21.05.2026

## Objective

This notebook transforms the raw GEO experiment data into a JSONL format compatible with Google Agent Search. For each of the 500 queries, one "universe" (i.e. experimental unit) is created per GEO editing dimension: each universe places the target document in its dimension-specific variant alongside the same 9 competitor documents. The resulting master JSONL file is used to populate the Agent Search index for retrieval-stage evaluation.

**Input files:**
- `competitor_docs.json` — 500 queries, each with 9 competitor documents and metadata
- `selected_docs.json` — target documents across all GEO editing dimensions

**Output:**
- `retail_geo_vertex_v1.jsonl` — one JSON record per document per universe, ready for Agent Search upload

## Data Transformation

In [ ]:
import json
import os
import re

# --- Configuration ---
COMPETITOR_JSON_PATH = os.path.join("..", "..", "data", "retail", "dataset", "1_preparation_competitor_docs_v1.json")
TARGETS_JSON_PATH = os.path.join("..", "..", "data", "retail", "dataset", "selected_docs.json")
MASTER_OUTPUT_FILE = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_vertex_geoedits_v1")

def build_parallel_universes():
    # 1. Load both JSON files
    with open(COMPETITOR_JSON_PATH, 'r', encoding='utf-8') as f_comp:
        competitor_data = json.load(f_comp)
        
    with open(TARGETS_JSON_PATH, 'r', encoding='utf-8') as f_target:
        selected_data = json.load(f_target)

    print(f"Loaded {len(competitor_data)} queries from competitor data.")
    print(f"Loaded {len(selected_data)} queries from target data.")
    print("Merging and building Master JSONL...\n")
    
    total_docs_written = 0
    missing_data_count = 0

    with open(MASTER_OUTPUT_FILE, 'w', encoding='utf-8') as out_file:
        
        # Loop through the root keys (0 to 499)
        for root_idx, comp_info in competitor_data.items():
            
            base_query_id = comp_info.get("query_id")
            query_text = comp_info.get("query")
            target_idx = int(comp_info.get("target_doc_idx")) 
            competitors = comp_info.get("competitor_docs", [])
            
            if len(competitors) != 9:
                print(f"Warning: Root index {root_idx} has {len(competitors)} competitors instead of 9.")

            try:
                targets_dict = selected_data[str(root_idx)][str(target_idx)]
            except KeyError:
                print(f"Error: Could not find targets for root_idx {root_idx} and target_idx {target_idx}. Skipping.")
                continue

            for raw_dimension_name, target_text in targets_dict.items():
                
                if not target_text:
                    print(f"Data Missing: Skipping '{raw_dimension_name}' for query '{base_query_id}'")
                    missing_data_count += 1
                    continue
                
                universe_id = f"{base_query_id}_{raw_dimension_name}"
                
                merged_docs = competitors.copy()
                merged_docs.insert(target_idx, target_text)

                for index, raw_doc in enumerate(merged_docs):
                    
                    if isinstance(raw_doc, dict):
                        final_text_payload = raw_doc.get("doc", "")
                    else:
                        final_text_payload = raw_doc
                    
                    safe_dimension_name = re.sub(r'[^a-zA-Z0-9-_]', '_', raw_dimension_name)
                    safe_universe_id = f"{base_query_id}_{safe_dimension_name}"
                    
                    if index == target_idx:
                        doc_id = f"{safe_universe_id}_TARGET"
                    else:
                        doc_id = f"{safe_universe_id}_comp_{index}"
                        
                    vertex_doc = {
                        "id": doc_id,
                        "structData": {
                            "query_id": universe_id,
                            "original_query": query_text, 
                            "text": final_text_payload 
                        }
                    }
                    
                    out_file.write(json.dumps(vertex_doc) + "\n")
                    total_docs_written += 1

    print(f"\n--- Process Complete ---")
    print(f"Master file created with {total_docs_written} total documents.")
    if missing_data_count > 0:
        print(f"Note: {missing_data_count} target variations were skipped due to missing text.")
    print("You can now upload this clean file to Vertex AI.")

build_parallel_universes()